In [29]:
import osmnx as ox
import networkx as nx
import folium

place = "District 1, Ho Chi Minh City, Vietnam"
G = ox.graph_from_place(place, network_type='drive')

kho = {
    "A": (10.7915, 106.6895),
    "B": (10.7715, 106.6985)
}

diem = {
    "1": (10.7824, 106.6958),
    "2": (10.7876, 106.7051),
    "3": (10.7650, 106.7020)
}

m = folium.Map(location=[10.78,106.69], zoom_start=14)

assign = {k: [] for k in kho}

for d_name, d_coord in diem.items():
    best_k = None
    best_len = 1e9
    d_node = ox.distance.nearest_nodes(G, d_coord[1], d_coord[0])

    for k_name, k_coord in kho.items():
        k_node = ox.distance.nearest_nodes(G, k_coord[1], k_coord[0])
        length = nx.shortest_path_length(G, k_node, d_node, weight='length')
        if length < best_len:
            best_len = length
            best_k = k_name

    assign[best_k].append((d_name, d_coord))

for k_name, k_coord in kho.items():
    folium.Marker(k_coord, popup="Kho " + k_name, icon=folium.Icon(color="green")).add_to(m)

for k_name, ds in assign.items():
    for d_name, d_coord in ds:
        k_node = ox.distance.nearest_nodes(G, kho[k_name][1], kho[k_name][0])
        d_node = ox.distance.nearest_nodes(G, d_coord[1], d_coord[0])

        path = nx.shortest_path(G, k_node, d_node, weight='length')
        coords = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in path]

        folium.PolyLine(coords).add_to(m)
        folium.Marker(d_coord, popup="Diem " + d_name).add_to(m)

m

**Nhận xét:**

Phương pháp tối ưu giúp mỗi điểm giao được gán về kho gần nhất,
từ đó giảm tổng quãng đường di chuyển.

So với cách không tối ưu (gán ngẫu nhiên), phương pháp này giúp tiết kiệm thời gian
và chi phí vận chuyển.

Kết quả cho thấy việc sử dụng thuật toán tìm đường ngắn nhất giúp cải thiện hiệu quả giao hàng.